## Filter-scale verification and closure discovery

Run top to bottom for one scale, or submit `slurm/run_filter_sweep.sbatch` for a Slurm array. The runner executes the `pace:*` tagged cells directly from this notebook; all recovery, model discovery, and scientific archiving remain here. See `PACE_FILTER_SWEEP.md` for submission, configuration, and result interpretation.

Order: configuration/data/filtering, source provenance, known momentum/heat checks and inhomogeneous recovery, shared first-order tau/phi fits, general SGS closures, second-order fits, archives/flux diagnostics, then plots. Inhomogeneous targets are found structurally (time derivative of u or b), with coefficient normalized to +1. No numeric column indices survive library changes. All candidate errors are training-data diagnostics; no automatic thesis selection or accuracy pass/fail is implied.


In [ ]:
import os
import json
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
import numpy as np
from SPIDER.continuous.library import Observable
from SPIDER.continuous.process_library_terms import SRDataset
from SPIDER.commons.library import Equation
from SPIDER.commons.z3base import FullRank, SymmetricTraceFree
from SPIDER.commons.sparse_reg_bf import Initializer, ModelIterator, Residual, Scaler, Threshold, sparse_reg_bf

# Batch runner injects RUN_CONFIG; interactive runs use these defaults.
analysis_config = {
    'data_path': '/storage/scratch1/2/ashelestov3/preprocessed_paper_data.npz',
    'delta_dx': 50., 'alpha': 3., 'seed': 1,
    'workers': int(os.environ.get('SLURM_CPUS_PER_TASK', '1')),
    'domain_width': 64, 'domain_time': 40, 'domains': 1000,
    'recovery_max_terms': 8, 'closure_max_terms': 6,
    'maximum_correction_terms': 5, 'tolerance': 1e-8,
    'Rayleigh': 1e8, 'Prandtl': 1., 'time_start': 0, 'time_stop': None,
    'output_root': 'thesis_runs',
}
analysis_config.update(globals().get('RUN_CONFIG', {}))
if any(analysis_config[k] <= 0 for k in ('delta_dx', 'alpha', 'workers', 'domains', 'domain_width', 'domain_time', 'Rayleigh', 'Prandtl')):
    raise ValueError('Filter, resource, integration, and physical parameters must be positive')
num_processors = int(analysis_config['workers'])
dom_width, dom_time = (int(analysis_config[k]) for k in ('domain_width', 'domain_time'))
ndomains = int(analysis_config['domains'])
analysis_run_dir = Path(analysis_config.get('run_dir') or
    (Path(analysis_config['output_root']) / (datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ') + '_' + uuid4().hex[:8])))
analysis_run_dir.mkdir(parents=True, exist_ok=True)

def analysis_json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    raise TypeError(type(value).__name__)

def write_analysis_report(name, report):
    (analysis_run_dir / name).write_text(json.dumps(report, indent=2, allow_nan=False,
        ensure_ascii=False, default=analysis_json_default), encoding='utf-8')

def integrate_analysis_dataset(dataset):
    if np.any(np.asarray(dataset.world_size) < [dom_width, dom_width, dom_time]):
        raise ValueError(f'Integration domain does not fit {dataset.world_size}')
    np.random.seed(analysis_config['seed'])
    dataset.make_domains(ndomains=ndomains, domain_size=[dom_width, dom_width, dom_time], pad=0)
    dataset.make_weights(m=16, qmax=0)
    dataset.set_LT_scale(L=dx*dom_width, T=dt*dom_time)
    dataset.cache_primes = num_processors > 1  # serial SPIDER does not clear per-domain caches
    dataset.make_library_matrices(debug=False, parallel=num_processors > 1, num_processors=num_processors)

def integration_metadata(dataset):
    return dict(world_size=dataset.world_size, dxs=dataset.dxs,
        domain_size=dataset.domain_size, length_scale=dataset.xscale, time_scale=dataset.tscale,
        domains=[dict(min=d.min_corner, max=d.max_corner) for d in dataset.domains],
        weights=[dict(m=w.m, q=w.q) for w in dataset.weights])


In [ ]:
load_path = str(analysis_config['data_path'])
print('Loading', load_path, flush=True)
with np.load(load_path) as data:
    selection = slice(analysis_config['time_start'], analysis_config['time_stop'])
    # Preprocessing swaps Dedalus velocity components before saving. Undo that swap.
    U = np.ascontiguousarray(data['V'][..., selection])
    V = np.ascontiguousarray(data['U'][..., selection])
    P = np.ascontiguousarray(data['P'][..., selection])
    B = np.ascontiguousarray(data['B'][..., selection])
    W = np.ascontiguousarray(data['W'][..., selection])
    x_raw, y_raw = np.asarray(data['x']), np.asarray(data['y'])
    t = np.asarray(data['t'][selection])
if U.shape != (len(x_raw), len(y_raw), len(t)) or any(a.shape != U.shape for a in (V, P, B, W)):
    raise ValueError('NPZ fields and coordinates are not aligned')
if len(t) < 3:
    raise ValueError('Need at least three time samples')
dx, dy, dt = (float(np.mean(np.diff(c))) for c in (x_raw, y_raw, t))
for name, coordinates, spacing in (('x', x_raw, dx), ('y', y_raw, dy), ('t', t, dt)):
    if spacing <= 0 or not np.isfinite(coordinates).all() or not np.allclose(np.diff(coordinates), spacing, rtol=1e-6, atol=1e-12):
        raise ValueError(f'{name} must be increasing and uniformly sampled; do not relabel coordinates')
x, y = x_raw.copy(), y_raw.copy()
Rayleigh, Prandtl = analysis_config['Rayleigh'], analysis_config['Prandtl']
kappa, nu = (Rayleigh*Prandtl)**(-.5), (Rayleigh/Prandtl)**(-.5)
delta, alpha = analysis_config['delta_dx']*dx, analysis_config['alpha']
print('Data shape:', U.shape, '| dx,dy,dt:', dx, dy, dt, '| delta:', delta, flush=True)


In [ ]:
from scipy.ndimage import convolve1d

def _get_uniform_gaussian_kernel(spacing, delta, alpha):
    radius = int(np.ceil(alpha*delta/spacing))
    offsets = np.arange(-radius, radius+1)*spacing
    kernel = np.exp(-.5*(offsets/(delta/np.sqrt(12)))**2)
    kernel[np.abs(offsets) > alpha*delta] = 0
    return kernel/kernel.sum()

def gaussian_filter_2d_finite_support_over_time(field, x, y, delta, alpha, multiplier=None):
    """Spatial Gaussian: periodic x; discard all wall-support points in y."""
    kernel_x = _get_uniform_gaussian_kernel(float(x[1]-x[0]), delta, alpha)
    kernel_y = _get_uniform_gaussian_kernel(float(y[1]-y[0]), delta, alpha)
    wall_pad = len(kernel_y)//2
    if 2*wall_pad >= len(y):
        raise ValueError('Filter support consumes the vertical domain')
    ys = slice(wall_pad, len(y)-wall_pad)
    result = np.empty((len(x), len(y)-2*wall_pad, field.shape[2]), dtype=float)
    for it in range(field.shape[2]):
        frame = np.asarray(field[:, :, it], dtype=float)
        if multiplier is not None:
            frame = frame * multiplier[:, :, it]
        frame = convolve1d(frame, kernel_x, axis=0, mode='wrap')
        frame = convolve1d(frame, kernel_y, axis=1, mode='nearest')
        result[:, :, it] = frame[:, ys]
    return result, np.asarray(x).copy(), np.asarray(y)[ys].copy()

def filter_field(field, multiplier=None):
    return gaussian_filter_2d_finite_support_over_time(field, x_raw, y_raw, delta, alpha, multiplier)[0]

u_filt, xp, yp = gaussian_filter_2d_finite_support_over_time(U, x_raw, y_raw, delta, alpha)
v_filt, p_filt, b_filt, w_filt = (filter_field(field) for field in (V, P, B, W))
u_bar = np.stack([u_filt, v_filt], axis=-1)
tau = np.empty(u_bar.shape[:3]+(2, 2), dtype=float)
tau[..., 0, 0] = filter_field(U, U)-u_filt*u_filt
tau[..., 1, 1] = filter_field(V, V)-v_filt*v_filt
tau[..., 0, 1] = filter_field(U, V)-u_filt*v_filt
tau[..., 1, 0] = tau[..., 0, 1]
phi = np.stack([filter_field(U, B)-u_filt*b_filt, filter_field(V, B)-v_filt*b_filt], axis=-1)
x, y = xp.copy(), yp.copy()
filter_metadata = dict(delta=delta, delta_dx=delta/dx, alpha=alpha, sigma=delta/np.sqrt(12),
    x_boundary='periodic wrap', y_boundary='nearest, then discard complete support',
    wall_pad=int(np.ceil(alpha*delta/dy)), pressure='ordinary filtered pressure', stress='full tau',
    kernel='sampled Gaussian, offsets outside alpha*delta set to zero, normalized sum',
    backend='scipy.ndimage.convolve1d per time frame', time_filter=False)
print('Filtered shape:', u_bar.shape, flush=True)


In [ ]:
# Shared source-provenance and archive helpers.
# For an existing kernel, no library construction/regression rerun is required.
from copy import deepcopy
import ast
import csv
import hashlib
import importlib.metadata
import inspect
import json
import operator
import os
import platform
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

archive_root = analysis_run_dir / "second_order_archives"
dedalus_source = Path(analysis_config.get("dedalus_source", "dedalus/paper-test/rayleigh_benard.py"))
dedalus_source_confirmed = False  # Set True only after verifying this generated your data.
source_h5_path = "/storage/scratch1/2/ashelestov3/snaps_paper/snaps_paper_s1.h5"
run_notes = ""
thesis_selected_model_ids = []  # Leave empty until you have reviewed/validated candidates.

def archive_json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    raise TypeError(f"Cannot archive {type(value).__name__}")

def git_metadata(*args):
    try:
        result = subprocess.run(["git", *args], capture_output=True, text=True, check=True)
        return result.stdout.strip()
    except (OSError, subprocess.CalledProcessError):
        return None

def file_identity(path):
    path = Path(path)
    info = {"path": str(path), "exists": path.is_file()}
    if info["exists"]:
        stat = path.stat()
        info.update(size_bytes=stat.st_size, mtime_ns=stat.st_mtime_ns)
    return info

def read_source_configuration(source):
    # Parse only: never import or execute a Dedalus simulation.
    tree = ast.parse(source)
    values, assignments = {}, []
    binary = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
              ast.Div: operator.truediv, ast.Pow: operator.pow}
    def scalar(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float, str)):
            return node.value
        if isinstance(node, ast.Name) and node.id in values:
            return values[node.id]
        if isinstance(node, ast.UnaryOp) and isinstance(node.op, (ast.USub, ast.UAdd)):
            return (-1 if isinstance(node.op, ast.USub) else 1) * scalar(node.operand)
        if isinstance(node, ast.BinOp) and type(node.op) in binary:
            return binary[type(node.op)](scalar(node.left), scalar(node.right))
        raise ValueError("Not a static scalar")

    wanted = {"Lx", "Lz", "Nx", "Nz", "Rayleigh", "Prandtl", "dealias", "dtype",
              "stop_sim_time", "timestepper", "max_timestep", "dt", "kappa", "nu"}
    for statement in tree.body:
        if not isinstance(statement, ast.Assign):
            continue
        for target in statement.targets:
            pairs = (zip(target.elts, statement.value.elts)
                     if isinstance(target, (ast.Tuple, ast.List))
                     and isinstance(statement.value, (ast.Tuple, ast.List))
                     else [(target, statement.value)])
            for name, expression in pairs:
                if isinstance(name, ast.Name):
                    try:
                        values[name.id] = scalar(expression)
                    except (ValueError, TypeError, ArithmeticError):
                        pass
                    if name.id in wanted:
                        assignments.append({"name": name.id, "line": statement.lineno,
                                            "expression": ast.unparse(expression),
                                            "static_value": values.get(name.id)})
    calls, time_updates = [], []
    for node in ast.walk(tree):
        if isinstance(node, ast.Call):
            function = ast.unparse(node.func)
            if (function.endswith(("add_equation", "load_state", "add_file_handler",
                                   "fill_random", "RealFourier", "ChebyshevT", "CFL"))
                    or function == "solver.step"):
                calls.append({"line": node.lineno, "expression": ast.unparse(node)})
        if isinstance(node, ast.Assign):
            for target in node.targets:
                if ast.unparse(target) in {"solver.stop_sim_time", "timestep"}:
                    time_updates.append({"line": node.lineno,
                                         "expression": ast.unparse(node)})
    return {"parameters": assignments, "equations_bases_initialization_and_output": calls,
            "effective_time_control_statements": time_updates,
            "provenance": "configuration parsed from current source; not a historical execution log"}


In [ ]:
# Archive inputs/settings/source before expensive recovery. Later stages add reports.
import shutil

analysis_provenance = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'validation_status': 'training-data verification; not a validated or thesis-selected model',
    'config': deepcopy(analysis_config), 'filter': filter_metadata,
    'data': {'npz': file_identity(load_path), 'raw_shape': list(U.shape), 'filtered_shape': list(u_bar.shape),
             'velocity_mapping': 'physical u_x=NPZ V, u_y=NPZ U; inverse of preprocessing swap',
             'coordinates': 'use saved x,y,t; no coordinate relabeling',
             'dt': dt, 'dt_min': float(np.diff(t).min()), 'dt_max': float(np.diff(t).max()),
             'content_hash': None, 'identity_note': 'file path/stat and coordinates; full NPZ not hashed'},
    'physics': {'Rayleigh': Rayleigh, 'Prandtl': Prandtl, 'nu': nu, 'kappa': kappa},
    'regression': {'inhomogeneous_target_coefficient': 1, 'train_fraction': 1., 'unit_rows': True,
        'initializer': 'power for multi-term models, combinatorial for two-column first order',
        'residual': 'matrix_relative', 'threshold': 'jump', 'gamma': 1.5, 'delta': 1e-6,
        'first_order_threshold_delta': 1e-8, 'max_passes': 20, 'seed': analysis_config['seed']},
    'git': {'commit': git_metadata('rev-parse', 'HEAD'), 'status': git_metadata('status', '--porcelain')},
    'sources': {}, 'dedalus': {'selected_source': str(dedalus_source),
        'source_confirmed_by_user': dedalus_source_confirmed, 'configurations': {}},
    'runtime': {'python': sys.version, 'platform': platform.platform(),
        'slurm': {name: os.environ.get(name) for name in ('SLURM_JOB_ID', 'SLURM_ARRAY_TASK_ID', 'SLURM_CPUS_PER_TASK')}},
    'packages': {},
}
for package in ('numpy', 'scipy', 'findiff', 'z3-solver'):
    try:
        analysis_provenance['packages'][package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        analysis_provenance['packages'][package] = None
source_paths = [Path('COnvection-FILTER-Copy2.ipynb'), Path('prepare-numpy-files-interpolate.ipynb'),
    dedalus_source, Path('run_filter_sweep.py'), Path('slurm/run_filter_sweep.sbatch')]
source_paths += sorted(Path('SPIDER').rglob('*.py'))
for index, path in enumerate(source_paths):
    identity = file_identity(path)
    if path.is_file():
        content = path.read_bytes()
        identity['sha256'] = hashlib.sha256(content).hexdigest()
        filename = f'input_source_{index}_{path.name}'
        (analysis_run_dir / filename).write_bytes(content)
        identity['archive_file'] = filename
        if path == dedalus_source:
            analysis_provenance['dedalus']['configurations'][str(path)] = read_source_configuration(content.decode('utf-8-sig'))
    analysis_provenance['sources'][str(path)] = identity
np.savez_compressed(analysis_run_dir / 'input_coordinates.npz', x_raw=x_raw, y_raw=y_raw, x=x, y=y, t=t)
write_analysis_report('provenance.json', analysis_provenance)


In [ ]:
# Shared weak library: recover momentum/heat with measured tau/phi available.
unit_vect = np.stack([np.zeros_like(b_filt), np.ones_like(b_filt)], axis=-1)
uobs, bobs, pobs = (Observable(string=name, rank=rank) for name, rank in (('u', 1), ('b', 0), ('p', 0)))
zobs = Observable(string='ez', rank=1)
Tobs = Observable(string='T', rank=2, can_commute_indices=True)
fobs = Observable(string='phi', rank=1)
srd = SRDataset(world_size=np.array(b_filt.shape),
    data_dict={'u': u_bar, 'b': b_filt, 'p': p_filt, 'ez': unit_vect, 'T': tau, 'phi': phi},
    observables=[uobs, bobs, pobs, zobs, Tobs, fobs], dxs=[dx, dy, dt],
    irreps=(FullRank(rank=0), FullRank(rank=1), SymmetricTraceFree(rank=2)))
srd.make_libraries(max_complexity=4, max_observables=2, max_dt=1, max_dx=2)
# Derivatives of the constant vertical unit vector are identically zero.
for lib in srd.libs.values():
    lib.terms = [term for term in lib.terms if not any(
        p.derivand.string == 'ez' and p.nderivs for p in term.primes)]
integrate_analysis_dataset(srd)

def recovery_term_kind(term, field):
    """Identify physical terms from index structure, never column numbers."""
    ps = term.primes
    if len(ps) == 1:
        p = ps[0]
        name, der = p.derivand.string, p.derivative
        xs = der.x_derivatives
        inds = p.derivand.indices
        if name == field and der.torder == 1 and der.xorder == 0:
            return 'time'
        if der.torder:
            return None
        if name == field and der.xorder == 2 and xs[0] == xs[1] and xs[0] not in inds:
            return 'diffusion'
        if field == 'u' and name == 'p' and der.xorder == 1:
            return 'pressure'
        if field == 'u' and name == 'T' and der.xorder == 1 and len(set(inds)) == 2 and xs[0] in inds:
            return 'sgs'
        if field == 'b' and name == 'phi' and der.xorder == 1 and xs[0] == inds[0]:
            return 'sgs'
    if len(ps) == 2:
        if field == 'u' and sorted(p.derivand.string for p in ps) == ['b', 'ez'] and all(p.nderivs == 0 for p in ps):
            return 'buoyancy'
        for velocity, differentiated in (ps, ps[::-1]):
            if (velocity.derivand.string == 'u' and velocity.nderivs == 0
                    and differentiated.derivand.string == field
                    and differentiated.derivative.torder == 0 and differentiated.derivative.xorder == 1
                    and differentiated.derivative.x_derivatives[0] == velocity.derivand.indices[0]
                    and (field == 'b' or differentiated.derivand.indices[0] != velocity.derivand.indices[0])):
                return 'advection'
    return None

def clean_regression_columns(lib, selected, target_col, priority=()):
    raw, weights = np.asarray(lib.Q, dtype=float), np.asarray(lib.col_weights, dtype=float)
    if not np.isfinite(raw).all() or not np.isfinite(weights).all() or np.any(weights <= 0):
        raise ValueError('Nonfinite library or invalid scales')
    scaled = raw[:, selected]/weights[selected]
    norms = {i: np.linalg.norm(scaled[:, j]) for j, i in enumerate(selected)}
    tolerance = analysis_config['tolerance']
    cutoff = tolerance*max(max(norms.values()), np.finfo(float).tiny)
    if norms[target_col] <= cutoff:
        raise ValueError(f'Numerically zero regression target: {lib.terms[target_col]}')
    order = list(dict.fromkeys([target_col, *priority, *selected]))
    keep, units, removed = [target_col], [], []
    for i in order:
        if i == target_col or i not in selected:
            continue
        if norms[i] <= cutoff:
            removed.append(dict(term=str(lib.terms[i]), reason='zero', original_column=i))
            continue
        unit = raw[:, i]/weights[i]/norms[i]
        if any(min(np.linalg.norm(unit-old), np.linalg.norm(unit+old)) <= tolerance for old in units):
            removed.append(dict(term=str(lib.terms[i]), reason='duplicate', original_column=i))
            continue
        keep.append(i)
        units.append(unit)
    Q = raw[:, keep]
    active = np.linalg.norm(Q/weights[keep], axis=1) > 0
    return Q[active], weights[keep], [lib.terms[i] for i in keep], removed

def inhomogeneous_candidates(Q, weights, terms, maximum_terms):
    """Column zero is the fixed target; archive coefficients normalized to +1."""
    max_k = min(int(maximum_terms), Q.shape[0], Q.shape[1])
    if max_k < 2:
        raise ValueError('Insufficient nonzero rows/predictors for inhomogeneous recovery')
    reports = []
    for count in range(2, max_k+1):
        result = sparse_reg_bf(Q,
            scaler=Scaler(sub_inds=None, char_sizes=weights, row_norms=None, unit_rows=True, train_fraction=1),
            initializer=Initializer(method='power', start_k=max_k),
            residual=Residual(residual_type='matrix_relative'),
            model_iterator=ModelIterator(max_k=max_k, backward_forward=True, max_passes=20),
            threshold=Threshold(threshold_type='jump', gamma=1.5, delta=1e-6, n_terms=count),
            inhomog=True, inhomog_col=0, verbose=False, seed=analysis_config['seed'])
        if result.xi[0] == 0 or not np.isfinite(result.xi).all():
            raise ValueError('Invalid inhomogeneous target coefficient')
        xi = result.xi/result.xi[0]
        reports.append(dict(requested_terms=count, retained_terms=int(np.count_nonzero(xi)),
            equation=' + '.join(f'({c:.12g}) * {term}' for term, c in zip(terms, xi) if c != 0)+' = 0',
            coefficients=[dict(term=str(term), coefficient=float(c)) for term, c in zip(terms, xi)],
            target_relative_error=float(np.linalg.norm(Q@xi)/np.linalg.norm(Q[:, 0])),
            spider_residual=float(result.lambd)))
        print(reports[-1]['equation'], '| error:', reports[-1]['target_relative_error'], flush=True)
    return reports

recovery_report = dict(validation_status='weak-form training fits; known-equation residuals are consistency checks',
    integration=integration_metadata(srd), library=dict(max_complexity=4, max_observables=2, max_dt=1, max_dx=2,
    observables=['u', 'b', 'p', 'ez', 'T', 'phi']), equations={})
for field, irrep, expected in (
    ('u', FullRank(rank=1), dict(time=1., advection=1., pressure=1., diffusion=-nu, buoyancy=-1., sgs=1.)),
    ('b', FullRank(rank=0), dict(time=1., advection=1., diffusion=-kappa, sgs=1.))):
    lib = srd.libs[irrep]
    physical = {}
    for kind in expected:
        matches = [i for i, term in enumerate(lib.terms) if recovery_term_kind(term, field) == kind]
        if len(matches) != 1:
            raise ValueError(f'{field}/{kind}: expected one physical term, found {matches}')
        physical[kind] = matches[0]
    target_col = physical['time']
    # Evaluate prescribed physics before any cleaning or regression.
    residual = sum(coefficient*lib.Q[:, physical[kind]] for kind, coefficient in expected.items())
    target_norm = np.linalg.norm(lib.Q[:, target_col])
    balance_scale = sum(abs(c)*np.linalg.norm(lib.Q[:, physical[k]]) for k, c in expected.items())
    known = dict(coefficients=[dict(term=str(lib.terms[physical[k]]), coefficient=c) for k, c in expected.items()],
        absolute_error=float(np.linalg.norm(residual)),
        target_relative_error=float(np.linalg.norm(residual)/target_norm) if target_norm else None,
        balance_relative_error=float(np.linalg.norm(residual)/balance_scale) if balance_scale else None,
        without_sgs_target_relative_error=float(np.linalg.norm(residual-lib.Q[:, physical['sgs']])/target_norm) if target_norm else None)
    print('Known equation:', field, known, flush=True)
    selected = [i for i, term in enumerate(lib.terms) if i == target_col or all(p.derivative.torder == 0 for p in term.primes)]
    Q, weights, terms, removed = clean_regression_columns(lib, selected, target_col, physical.values())
    candidates = inhomogeneous_candidates(Q, weights, terms, analysis_config['recovery_max_terms'])
    recovery_report['equations'][field] = dict(target=str(terms[0]), known=known, removed_columns=removed,
        retained_library=[str(term) for term in terms], column_weights=weights, candidates=candidates)
write_analysis_report('recovery.json', recovery_report)


In [ ]:
# Shared first-order tau/phi construction and regression, evaluated only once.
# The wall-support crop was already applied by the filter. This extra point is
# solely a derivative halo; no filter-scale-dependent double trimming.
pad_x = pad_y = 1
fit_crop = (slice(1, -1), slice(1, -1))
tau_fit, phi_fit = tau[fit_crop], phi[fit_crop]
resolved_fit = {'u': u_bar[fit_crop], 'p': p_filt[fit_crop], 'b': b_filt[fit_crop]}
grad_u_grad_u, grad_u_grad_b = np.empty_like(tau_fit), np.empty_like(phi_fit)
for it in range(len(t)):
    du_dx, du_dy = np.gradient(u_bar[:, :, it], x, y, axis=(0, 1), edge_order=2)
    db_dx, db_dy = np.gradient(b_filt[:, :, it], x, y, axis=(0, 1), edge_order=2)
    grad_u_grad_u[:, :, it] = (np.einsum('...i,...j->...ij', du_dx, du_dx)
        + np.einsum('...i,...j->...ij', du_dy, du_dy))[fit_crop]
    grad_u_grad_b[:, :, it] = (du_dx*db_dx[..., None]+du_dy*db_dy[..., None])[fit_crop]

def bare_observable(term, name):
    return len(term.primes) == 1 and term.primes[0].derivand.string == name and term.primes[0].nderivs == 0

def first_order_dataset(target, measured, gradient_name, gradient, rank, irreps):
    observables = [Observable(string=name, rank=rank, can_commute_indices=rank == 2)
                   for name in (target, gradient_name)]
    dataset = SRDataset(world_size=np.array(measured.shape[:3]),
        data_dict={target: measured, gradient_name: gradient}, observables=observables,
        dxs=[dx, dy, dt], irreps=irreps)
    dataset.make_libraries(max_complexity=rank, max_observables=1, max_dx=0, max_dt=0)
    integrate_analysis_dataset(dataset)
    results, blocks = {}, []
    for irrep, lib in dataset.libs.items():
        target_col, gradient_col = [next(i for i, term in enumerate(lib.terms) if bare_observable(term, name))
                                    for name in (target, gradient_name)]
        Q = np.asarray(lib.Q, dtype=float)
        if not np.isfinite(Q).all() or any(np.linalg.norm(Q[:, col]) == 0 for col in (target_col, gradient_col)):
            raise ValueError(f'{target}/{irrep}: nonfinite or zero first-order columns')
        active = np.linalg.norm(Q, axis=1) > 0
        result = sparse_reg_bf(Q[active],
            scaler=Scaler(sub_inds=None, char_sizes=lib.col_weights, row_norms=None, unit_rows=True, train_fraction=1),
            initializer=Initializer(method='combinatorial', start_k=2),
            residual=Residual(residual_type='matrix_relative'),
            model_iterator=ModelIterator(max_k=2, backward_forward=False, max_passes=1),
            threshold=Threshold(threshold_type='jump', gamma=1.5, delta=1e-8, n_terms=2),
            inhomog=True, inhomog_col=target_col, verbose=False, seed=analysis_config['seed'])
        if result.xi[target_col] == 0 or not np.isfinite(result.xi).all():
            raise ValueError('Invalid first-order fit')
        coefficient = float(-result.xi[gradient_col]/result.xi[target_col])
        measured_column, gradient_column = lib.Q[:, target_col], lib.Q[:, gradient_col]
        denominator = np.linalg.norm(measured_column)
        results[str(irrep)] = dict(fitted_coefficient=coefficient,
            equation=f'{lib.terms[target_col]} = ({coefficient:.12g}) * {lib.terms[gradient_col]}',
            target_relative_residual=float(np.linalg.norm(measured_column-coefficient*gradient_column)/denominator)
                if denominator else None)
        blocks.append((measured_column, gradient_column))
    return dataset, results, blocks

tau_first_order_srd, tau_first_order_results, tau_blocks = first_order_dataset(
    'tau', tau_fit, 'grad_u_grad_u', grad_u_grad_u, 2, (FullRank(rank=0), SymmetricTraceFree(rank=2)))
phi_first_order_srd, phi_results, _ = first_order_dataset(
    'phi', phi_fit, 'grad_u_grad_b', grad_u_grad_b, 1, (FullRank(rank=1),))
phi_first_order_results = phi_results[str(FullRank(rank=1))]
joint_tau, joint_gradient = (np.concatenate([block[i] for block in tau_blocks]) for i in (0, 1))
if not np.dot(joint_gradient, joint_gradient) > 0:
    raise ValueError('Zero gradient baseline')
combined_coefficient = float(np.dot(joint_gradient, joint_tau)/np.dot(joint_gradient, joint_gradient))
combined_weak_residual = float(np.linalg.norm(joint_tau-combined_coefficient*joint_gradient)/np.linalg.norm(joint_tau))
expected_coefficient = delta**2/12
first_order_report = dict(expected_coefficient=expected_coefficient, tau_per_irrep=tau_first_order_results,
    tau_combined=dict(fitted_coefficient=combined_coefficient, weak_error=combined_weak_residual),
    phi=phi_first_order_results, pointwise={}, validation_status='training fits; no independent validation')
for target, measured, gradient, fitted in (
        ('tau', tau_fit, grad_u_grad_u, combined_coefficient),
        ('phi', phi_fit, grad_u_grad_b, phi_first_order_results['fitted_coefficient'])):
    errors = np.zeros(3)
    for it in range(len(t)):
        a, g = measured[:, :, it], gradient[:, :, it]
        errors += [np.sum((a-fitted*g)**2), np.sum((a-expected_coefficient*g)**2), np.sum(a*a)]
    first_order_report['pointwise'][target] = dict(fitted=float(np.sqrt(errors[0]/errors[2])) if errors[2] else None,
        theoretical=float(np.sqrt(errors[1]/errors[2])) if errors[2] else None)
first_order_report['integration'] = {name: integration_metadata(dataset) for name, dataset in
    (('tau', tau_first_order_srd), ('phi', phi_first_order_srd))}
write_analysis_report('first_order.json', first_order_report)
print(first_order_report['pointwise'])


In [ ]:
# Unconstrained resolved-field closures from the same shared library.
# Restrict predictors to spatial terms in u,p,b; measured SGS fields are targets only.
shared_closure_report = {}
for target, irreps in ((Tobs.string, (FullRank(rank=0), SymmetricTraceFree(rank=2))),
                      (fobs.string, (FullRank(rank=1),))):
    for irrep in irreps:
        lib = srd.libs[irrep]
        matches = [i for i, term in enumerate(lib.terms) if bare_observable(term, target)]
        if len(matches) != 1:
            raise ValueError(f'Expected one bare {target} target')
        target_col = matches[0]
        selected = [i for i, term in enumerate(lib.terms) if i == target_col or all(
            p.derivand.string in {'u', 'p', 'b'} and p.derivative.torder == 0 for p in term.primes)]
        Q, weights, terms, removed = clean_regression_columns(lib, selected, target_col)
        shared_closure_report[f'{target}|{irrep}'] = dict(removed_columns=removed,
            retained_library=[str(term) for term in terms], column_weights=weights,
            candidates=inhomogeneous_candidates(Q, weights, terms, analysis_config['closure_max_terms']))
write_analysis_report('shared_closures.json', shared_closure_report)


### Second-order SGS discovery

Fit the remainder R_tau = tau - C_tau G_tau or R_phi = phi - C_phi G_phi with the same inhomogeneous SPIDER regression used above. Run the shared first-order discovery cell first. Both fitted and theoretical coefficients are frozen; the reported correction is added to that baseline.

The complexity-6 resolved-field library includes products of second spatial derivatives. All columns are evaluated together on each first-order dataset's domains and weights. Results are training fits, not independent validation. Numerical execution belongs on PACE.


In [ ]:
"""Residual SGS discovery helpers for the PACE convection notebook."""

from copy import deepcopy
from dataclasses import dataclass

import numpy as np

from SPIDER.commons.library import Observable
from SPIDER.commons.sparse_reg_bf import (
    Initializer, ModelIterator, Residual, Scaler, Threshold, sparse_reg_bf,
)
from SPIDER.continuous.process_library_terms import SRDataset


def bare_column(lib, name):
    columns = [i for i, term in enumerate(lib.terms)
               if len(term.primes) == 1
               and term.primes[0].derivand.string == name
               and term.primes[0].nderivs == 0]
    if len(columns) != 1:
        raise ValueError(f"Expected one bare {name} column, found {columns}")
    return columns[0]


def build_correction_dataset(first_order_srd, resolved_data, *,
                             num_processors, parallel=True):
    """Evaluate resolved predictors and baseline columns on the same rows.

    resolved_data must use exactly the crop of first_order_srd, not merely
    arrays with a coincidentally matching shape. No new domains are sampled.
    """
    if set(resolved_data) != {"u", "p", "b"}:
        raise ValueError("Expected resolved fields u, p, b")
    for name, values in resolved_data.items():
        shape = tuple(first_order_srd.world_size) + ((2,) if name == "u" else ())
        if values.shape != shape:
            raise ValueError(f"{name}: expected aligned shape {shape}, got {values.shape}")
    observables = [Observable(string="u", rank=1),
                   Observable(string="p", rank=0),
                   Observable(string="b", rank=0)]
    dataset = SRDataset(
        world_size=np.array(first_order_srd.world_size, copy=True),
        data_dict={**resolved_data, **first_order_srd.data_dict},
        observables=observables, dxs=list(first_order_srd.dxs),
        irreps=first_order_srd.irreps,
    )
    # Two observables plus four spatial derivatives admit Hessian products.
    dataset.make_libraries(max_complexity=6, max_observables=2,
                           max_dx=4, max_dt=0)
    for irrep, lib in dataset.libs.items():
        # Only append the undifferentiated measured and first-order fields.
        # They never participate in generated products or derivatives.
        source_terms = list(first_order_srd.libs[irrep].terms)
        if any(len(term.primes) != 1 or term.primes[0].nderivs != 0
               for term in source_terms):
            raise ValueError("Expected a two-field first-order dataset")
        lib.terms = list(lib.terms) + source_terms
    dataset.observables += list(first_order_srd.observables)
    dataset.domains = deepcopy(first_order_srd.domains)
    dataset.domain_size = list(first_order_srd.domain_size)
    dataset.pad = first_order_srd.pad
    dataset.weights, dataset.tensor_weight_basis = deepcopy(
        (first_order_srd.weights, first_order_srd.tensor_weight_basis))
    dataset.weight_dxs = list(first_order_srd.weight_dxs)
    dataset.set_LT_scale(first_order_srd.xscale, first_order_srd.tscale)
    dataset.cache_primes = parallel
    dataset.make_library_matrices(parallel=parallel,
                                  num_processors=num_processors, debug=False)
    return dataset


In [ ]:
def is_gradient_baseline(term, target_name):
    """Recognize the resolved equivalent of the explicitly evaluated G field."""
    primes = term.primes
    names = sorted(prime.derivand.string for prime in primes)
    expected = ["u", "u"] if target_name == "tau" else ["b", "u"]
    if names != expected or any(prime.derivative.torder != 0
                               or prime.derivative.xorder != 1 for prime in primes):
        return False
    left, right = (prime.derivative.x_derivatives[0] for prime in primes)
    observable_indices = [index for prime in primes for index in prime.derivand.indices]
    return left == right and left not in observable_indices


@dataclass
class ResidualProblem:
    Q: np.ndarray
    weights: np.ndarray
    terms: list
    measured: np.ndarray
    baseline: np.ndarray
    coefficient: float
    target_name: str
    gradient_name: str
    gradient_label: str
    removed: list
    residual_is_zero: bool


def residual_problem(lib, target_name, gradient_name, coefficient, *,
                     include_first_order_terms=False, tolerance=1e-8):
    """Subtract a frozen baseline in physical units, then scale and clean.

    Column zero always represents R = target - coefficient * gradient.
    Its term object is retained only for tensor indexing; reports label it R.
    """
    coefficient = float(coefficient)
    if not np.isfinite(coefficient) or not 0 < tolerance < 1:
        raise ValueError("Invalid baseline coefficient or tolerance")
    target_col = bare_column(lib, target_name)
    gradient_col = bare_column(lib, gradient_name)
    raw = np.asarray(lib.Q, dtype=float)
    weights = np.asarray(lib.col_weights, dtype=float)
    if (not np.isfinite(raw).all() or not np.isfinite(weights).all()
            or np.any(weights <= 0)):
        raise ValueError("Nonfinite library or invalid characteristic sizes")
    measured = raw[:, target_col].copy()
    baseline = coefficient * raw[:, gradient_col]
    residual = measured - baseline
    residual_norm = np.linalg.norm(residual)
    reference_norm = max(np.linalg.norm(measured), np.linalg.norm(baseline))
    is_zero = residual_norm <= tolerance * max(reference_norm, np.finfo(float).tiny)
    # Keep the original target's physical units, but scale its magnitude to R.
    residual_weight = weights[target_col] * (
        residual_norm / max(np.linalg.norm(measured), np.finfo(float).tiny))
    residual_weight = max(residual_weight, np.finfo(float).tiny)
    selected = []
    removed = []
    for i, term in enumerate(lib.terms):
        if i == target_col:
            continue
        if i == gradient_col or is_gradient_baseline(term, target_name):
            if not include_first_order_terms:
                removed.append((str(term), "frozen first-order term"))
                continue
        selected.append(i)
    scaled = raw[:, selected] / weights[selected]
    norms = np.linalg.norm(scaled, axis=0)
    cutoff = tolerance * max(norms.max(initial=0), np.finfo(float).tiny)
    keep, units = [], []
    for i, norm, column in zip(selected, norms, scaled.T):
        if norm <= cutoff:
            removed.append((str(lib.terms[i]), "zero"))
            continue
        unit = column / norm
        if any(min(np.linalg.norm(unit - old), np.linalg.norm(unit + old))
               <= tolerance for old in units):
            removed.append((str(lib.terms[i]), "duplicate"))
            continue
        keep.append(i)
        units.append(unit)
    return ResidualProblem(
        Q=np.column_stack([residual, raw[:, keep]]),
        weights=np.concatenate([[residual_weight], weights[keep]]),
        terms=[lib.terms[target_col]] + [lib.terms[i] for i in keep],
        measured=measured, baseline=baseline, coefficient=coefficient,
        target_name=target_name, gradient_name=gradient_name,
        gradient_label=str(lib.terms[gradient_col]),
        removed=removed, residual_is_zero=is_zero,
    )


In [ ]:
def discover_corrections(problem, *, maximum_correction_terms=5, seed=1):
    """Fit R with a fixed target coefficient and return physical coefficients.

    The zero-correction model is included explicitly. SPIDER's n_terms counts
    the forced residual column as one term in addition to the predictors.
    All rows are used, matching first-order discovery; these are training fits.
    """
    if maximum_correction_terms < 1:
        raise ValueError("maximum_correction_terms must be positive")
    n_predictors = problem.Q.shape[1] - 1
    fits = [{"correction_terms": 0, "coefficients": np.zeros(n_predictors),
             "result": None}]
    if problem.residual_is_zero or n_predictors == 0:
        return fits
    scaled = problem.Q / problem.weights
    # Avoid Scaler's division by zero on rows where every column vanishes.
    active = np.linalg.norm(scaled, axis=1) > 0
    Q = problem.Q[active]
    max_k = min(maximum_correction_terms + 1, Q.shape[1], Q.shape[0])
    if max_k < 2:
        raise ValueError("Insufficient nonzero rows for correction discovery")
    for n_terms in range(2, max_k + 1):
        result = sparse_reg_bf(
            Q,
            scaler=Scaler(sub_inds=None, char_sizes=problem.weights,
                          row_norms=None, unit_rows=True, train_fraction=1),
            initializer=Initializer(method="power", start_k=max_k),
            residual=Residual(residual_type="matrix_relative"),
            model_iterator=ModelIterator(max_k=max_k, backward_forward=True,
                                         max_passes=20),
            threshold=Threshold(threshold_type="jump", gamma=1.5,
                                delta=1e-6, n_terms=n_terms),
            inhomog=True, inhomog_col=0, verbose=False, seed=seed,
        )
        if not np.isfinite(result.xi).all() or result.xi[0] == 0:
            raise ValueError("Regression did not retain a finite residual target")
        coefficients = -result.xi[1:] / result.xi[0]
        fits.append({"correction_terms": int(np.count_nonzero(coefficients)),
                     "coefficients": coefficients, "result": result})
    return fits


In [ ]:
def summarize_corrections(problem, fits):
    """JSON-ready equations and errors on the original, unscaled weak rows."""
    measured_norm = float(np.linalg.norm(problem.measured))
    residual_norm = float(np.linalg.norm(problem.Q[:, 0]))

    def ratio(numerator, denominator):
        return float(numerator / denominator) if denominator > 0 else None

    reports = []
    for fit in fits:
        coefficients = fit["coefficients"]
        correction = problem.Q[:, 1:] @ coefficients
        error_norm = float(np.linalg.norm(problem.Q[:, 0] - correction))
        terms = [{"term": str(term), "coefficient": float(coefficient)}
                 for term, coefficient in zip(problem.terms[1:], coefficients)
                 if coefficient != 0]
        correction_text = " + ".join(
            f"({entry['coefficient']:.12g}) * {entry['term']}" for entry in terms
        ) or "0"
        target_label = str(problem.terms[0])
        baseline_text = f"({problem.coefficient:.12g}) * {problem.gradient_label}"
        reports.append({
            "correction_terms": fit["correction_terms"],
            "first_order_coefficient": problem.coefficient,
            "first_order_term": problem.gradient_label,
            "correction_coefficients": terms,
            "correction_model": f"R[{target_label}] = {correction_text}",
            "complete_model": f"{target_label} = {baseline_text} + {correction_text}",
            "baseline_relative_error": ratio(residual_norm, measured_norm),
            "corrected_relative_error": ratio(error_norm, measured_norm),
            "residual_relative_error": ratio(error_norm, residual_norm),
            "fractional_improvement": (1 - error_norm / residual_norm
                                       if residual_norm > 0 else None),
            "absolute_weak_error": error_norm,
            "spider_residual": (float(fit["result"].lambd)
                                if fit["result"] is not None else None),
            "baseline_remainder_numerically_zero": problem.residual_is_zero,
        })
    return reports


In [ ]:
# Explicit first-order model selection. "per_irrep" uses separate tau fits.
tau_baseline_selection = "combined"  # "combined" or "per_irrep"
baseline_modes = ("fitted", "theoretical")
include_first_order_terms = False  # True also allows coefficient corrections.
maximum_correction_terms = int(analysis_config["maximum_correction_terms"])
second_order_seed = int(analysis_config["seed"])
second_order_tolerance = float(analysis_config["tolerance"])

if tau_baseline_selection not in {"combined", "per_irrep"}:
    raise ValueError("Unknown tau baseline selection")
if not baseline_modes or not set(baseline_modes) <= {"fitted", "theoretical"}:
    raise ValueError("Choose fitted and/or theoretical baselines")

# Match the crop used to construct tau_fit, phi_fit and their gradient fields.
resolved_fit = {
    "u": u_bar[pad_x:-pad_x, pad_y:-pad_y],
    "p": p_filt[pad_x:-pad_x, pad_y:-pad_y],
    "b": b_filt[pad_x:-pad_x, pad_y:-pad_y],
}
first_order_sources = {
    "tau": (tau_first_order_srd, "grad_u_grad_u"),
    "phi": (phi_first_order_srd, "grad_u_grad_b"),
}
frozen_coefficients = {}
for target, (source, gradient_name) in first_order_sources.items():
    for irrep in source.irreps:
        if target == "phi":
            fitted = phi_first_order_results["fitted_coefficient"]
        elif tau_baseline_selection == "combined":
            fitted = combined_coefficient
        else:
            fitted = tau_first_order_results[str(irrep)]["fitted_coefficient"]
        frozen_coefficients[target, str(irrep)] = {
            "fitted": float(fitted), "theoretical": float(delta**2 / 12),
        }

# Expensive PACE step: evaluate once, then reuse for both coefficient choices.
second_order_datasets = {}
for target, (source, gradient_name) in first_order_sources.items():
    print("Building second-order library:", target, flush=True)
    second_order_datasets[target] = build_correction_dataset(
        source, resolved_fit, num_processors=num_processors, parallel=num_processors > 1,
    )


In [ ]:
# This cell can be rerun with different sparsity/baseline settings without
# repeating the expensive library integration.
# Freeze configuration with the results so later notebook edits cannot relabel a run.
second_order_run_settings = {
    "tau_baseline_selection": tau_baseline_selection,
    "baseline_modes": list(baseline_modes),
    "include_first_order_terms": include_first_order_terms,
    "maximum_correction_terms": maximum_correction_terms,
    "seed": second_order_seed, "tolerance": second_order_tolerance,
    "delta": float(delta), "alpha": float(alpha),
    "pad_x": int(pad_x), "pad_y": int(pad_y),
    "dx": float(dx), "dy": float(dy), "dt": float(dt),
    "load_path": str(load_path),
    "analysis_Rayleigh": float(Rayleigh), "analysis_Prandtl": float(Prandtl),
    "error_space": "unscaled weak-form training rows",
}
second_order_results = {}
for target, dataset in second_order_datasets.items():
    gradient_name = first_order_sources[target][1]
    for irrep, lib in dataset.libs.items():
        for mode in baseline_modes:
            key = (target, str(irrep), mode)
            problem = residual_problem(
                lib, target, gradient_name,
                frozen_coefficients[target, str(irrep)][mode],
                include_first_order_terms=include_first_order_terms,
                tolerance=second_order_tolerance,
            )
            print(key, "| frozen C =", problem.coefficient,
                  "| predictors =", len(problem.terms) - 1, flush=True)
            if problem.removed:
                print("Excluded columns:", problem.removed)
            if problem.residual_is_zero:
                print("Baseline remainder is numerically zero; skipping regression.")
            fits = discover_corrections(
                problem, maximum_correction_terms=maximum_correction_terms,
                seed=second_order_seed,
            )
            second_order_results[key] = {"problem": problem, "fits": fits,
                                         "run_settings": deepcopy(second_order_run_settings)}


In [ ]:
import json
from pathlib import Path

second_order_report = {
    "settings": {
        "tau_baseline_selection": tau_baseline_selection,
        "baseline_modes": list(baseline_modes),
        "include_first_order_terms": include_first_order_terms,
        "maximum_correction_terms": maximum_correction_terms,
        "seed": second_order_seed,
        "tolerance": second_order_tolerance,
        "delta": float(delta),
        "error_space": "unscaled weak-form training rows",
    },
    "integration": {},
    "models": [],
}
for target, dataset in second_order_datasets.items():
    second_order_report["integration"][target] = {
        "world_size": np.asarray(dataset.world_size).tolist(),
        "dxs": [float(value) for value in dataset.dxs],
        "domain_size": list(dataset.domain_size),
        "domains": [{"min": list(map(int, domain.min_corner)),
                     "max": list(map(int, domain.max_corner))}
                    for domain in dataset.domains],
        "weights": [{"m": list(map(int, weight.m)),
                     "q": list(map(int, weight.q))} for weight in dataset.weights],
        "length_scale": float(dataset.xscale),
        "time_scale": float(dataset.tscale),
    }

for (target, irrep, mode), entry in second_order_results.items():
    problem = entry["problem"]
    reports = summarize_corrections(problem, entry["fits"])
    entry["reports"] = reports
    print("\n", target, "|", irrep, "|", mode)
    print("terms | baseline/target | corrected/target | corrected/remainder")
    for report in reports:
        print(report["correction_terms"],
              report["baseline_relative_error"],
              report["corrected_relative_error"],
              report["residual_relative_error"], sep=" | ")
        print("  ", report["correction_model"])
        print("  ", report["complete_model"])
        second_order_report["models"].append({
            "target": target, "irrep": irrep, "baseline_mode": mode,
            "excluded_columns": problem.removed, **report,
        })

def report_json_default(value):
    # Convert NumPy scalars (including bool_) without changing their types.
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(f"Object of type {type(value).__name__} is not JSON serializable")

report_path = analysis_run_dir / "second_order_results.json"
report_path.write_text(
    json.dumps(second_order_report, indent=2, ensure_ascii=False, allow_nan=False,
               default=report_json_default),
    encoding="utf-8",
)
print("\nSaved", report_path.resolve())


In [ ]:
created = datetime.now(timezone.utc)
run_id = created.strftime("%Y%m%dT%H%M%S.%fZ") + "_" + uuid4().hex[:8]
bundle = {
    "schema_version": 1, "run_id": run_id, "created_utc": created.isoformat(),
    "notes": run_notes, "validation_status": "training fits; no independent validation recorded",
    "git": {"commit": git_metadata("rev-parse", "HEAD"),
            "working_tree_status_before_archive": git_metadata("status", "--porcelain")},
    "runtime": {"python": sys.version, "platform": platform.platform(),
                "slurm": {key: os.environ.get(key) for key in
                          ("SLURM_JOB_ID", "SLURM_CPUS_PER_TASK", "SLURM_NTASKS")}},
    "packages": {}, "models": [], "sources": {},
    "verification": {"provenance": analysis_provenance, "recovery": recovery_report,
                     "first_order": first_order_report, "shared_closures": shared_closure_report},
    "integration": second_order_report["integration"],
    "dedalus": {"selected_source": str(dedalus_source),
                "source_confirmed_by_user": dedalus_source_confirmed,
                "source_h5": file_identity(source_h5_path),
                "source_h5_provenance": "configured here; confirm against preprocessing input",
                "configurations": {}},
    "data": {"npz": file_identity(load_path),
             "content_hash": None,
             "identity_note": "path/stat plus archived coordinates; large field arrays are not hashed",
             "raw_velocity_shape": list(U.shape),
             "filtered_velocity_shape": list(u_bar.shape),
             "coordinate_convention": "analysis (x,y,t); Dedalus vertical coordinate is z",
             "dt_used": float(dt),
             "dt_from_t": {"min": float(np.min(np.diff(t))),
                           "max": float(np.max(np.diff(t))),
                           "mean": float(np.mean(np.diff(t)))},
             "time_range": [float(t[0]), float(t[-1])]},
    "current_analysis_parameters": {
        "Rayleigh": float(Rayleigh), "Prandtl": float(Prandtl),
        "kappa": float(kappa), "delta": float(delta), "alpha": float(alpha),
        "pad_x": int(pad_x), "pad_y": int(pad_y),
        "filter": filter_metadata,
        "spatial_filter_only": True,
        "capture_note": "current kernel values; compare with per-model frozen run_settings"},
    "library": {"max_complexity": 6, "max_observables": 2, "max_dx": 4, "max_dt": 0,
                "resolved_fields": ["u", "p", "b"],
                "capture_note": "defaults of archived implementation; executed helper source saved below"},
    "regression": {"method": "sparse_reg_bf", "inhomog": True, "target_column": 0,
                   "unit_rows": True, "train_fraction": 1,
                   "initializer": "power", "backward_forward": True, "max_passes": 20,
                   "residual": "matrix_relative", "threshold": "jump",
                   "gamma": 1.5, "delta": 1e-6,
                   "capture_note": "defaults of archived implementation; executed helper source saved below"},
    "paper": {"file": "2D_Incompressible_SGS_Paper_2 (44).pdf",
              "stress_reference": "section 3.1 equations (3.1)-(3.2); B6; C4",
              "C1_theoretical": float(delta**2 / 12),
              "C2_theoretical": float(delta**4 / 288)},
}
for package in ("numpy", "scipy", "jax", "dedalus", "findiff", "z3-solver"):
    try:
        bundle["packages"][package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        bundle["packages"][package] = None

for (target, irrep, mode), entry in second_order_results.items():
    settings = entry.get("run_settings", second_order_report["settings"])
    for index, model in enumerate(summarize_corrections(entry["problem"], entry["fits"])):
        model_id = f"{target}|{irrep}|{mode}|candidate={index}"
        bundle["models"].append({
            "model_id": model_id, "target": target, "irrep": irrep, "baseline_mode": mode,
            "selected_for_thesis": model_id in thesis_selected_model_ids,
            "run_settings": settings,
            "settings_provenance": ("captured when regression ran" if "run_settings" in entry
                                    else "reconstructed from report; pre-archive run, verify settings"),
            "excluded_columns": entry["problem"].removed, **model,
        })
unknown_ids = set(thesis_selected_model_ids) - {model["model_id"] for model in bundle["models"]}
if unknown_ids:
    raise ValueError(f"Unknown thesis selections: {unknown_ids}")

# Capture the actual function definitions in this kernel where IPython exposes them.
bundle["executed_helpers"] = {}
for name in ("build_correction_dataset", "residual_problem", "discover_corrections",
             "summarize_corrections", "gaussian_filter_2d_finite_support_over_time",
             "_get_uniform_gaussian_kernel", "filter_field"):
    try:
        bundle["executed_helpers"][name] = inspect.getsource(globals()[name])
    except (OSError, TypeError, KeyError):
        bundle["executed_helpers"][name] = None

run_dir = archive_root / run_id
run_dir.mkdir(parents=True, exist_ok=False)
for input_source in analysis_run_dir.glob('input_source_*'):
    (run_dir / input_source.name).write_bytes(input_source.read_bytes())
(run_dir / 'input_coordinates.npz').write_bytes((analysis_run_dir / 'input_coordinates.npz').read_bytes())
simulation_sources = [dedalus_source, Path("dedalus/paper-test/rayleigh_benard-stabilize.py"),
                      Path("dedalus/paper-test/rayleigh_benard-setup.py")]
sources = simulation_sources + [Path("prepare-numpy-files-interpolate.ipynb"),
                                Path("COnvection-FILTER-Copy2.ipynb")]
for index, path in enumerate(sources):
    identity = file_identity(path)
    if path.is_file():
        content = path.read_bytes()
        identity["sha256"] = hashlib.sha256(content).hexdigest()
        if path.suffix == ".ipynb":
            notebook = json.loads(content)
            # Source/metadata only; avoid duplicating embedded figures in every archive.
            snapshot = {"cells": [{"id": cell.get("id"), "cell_type": cell["cell_type"],
                                    "source": cell["source"]} for cell in notebook["cells"]]}
            filename = f"source_{index}_{path.stem}.json"
            (run_dir / filename).write_text(json.dumps(snapshot, indent=2), encoding="utf-8")
            identity["snapshot_note"] = "on-disk cell source; save notebook before archiving"
        else:
            filename = f"source_{index}_{path.name}"
            (run_dir / filename).write_bytes(content)
            bundle["dedalus"]["configurations"][str(path)] = read_source_configuration(
                content.decode("utf-8-sig"))
        identity["archive_file"] = filename
    bundle["sources"][str(path)] = identity

paper_path = Path(bundle["paper"]["file"])
if paper_path.is_file():
    bundle["paper"]["sha256"] = hashlib.sha256(paper_path.read_bytes()).hexdigest()
np.savez_compressed(run_dir / "coordinates.npz", x=x, y=y, t=t, xp=xp, yp=yp)
# Strict JSON keeps booleans/numbers numeric and rejects nonfinite values.
(run_dir / "run.json").write_text(
    json.dumps(bundle, indent=2, ensure_ascii=False, allow_nan=False, default=archive_json_default),
    encoding="utf-8",
)
with (run_dir / "models.csv").open("w", newline="", encoding="utf-8") as stream:
    fields = ["model_id", "target", "irrep", "baseline_mode", "correction_terms",
              "first_order_coefficient", "complete_model", "baseline_relative_error",
              "corrected_relative_error", "residual_relative_error", "selected_for_thesis"]
    writer = csv.DictWriter(stream, fieldnames=fields, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(bundle["models"])
equations = ["# Candidate models", "", f"Run: {run_id}", "",
             "Errors below are unscaled weak-form training errors.", ""]
for model in bundle["models"]:
    equations.extend([f"## {model['model_id']}", "", model["complete_model"], "",
                      f"Relative error: {model['corrected_relative_error']}", ""])
(run_dir / "models.md").write_text("\n".join(equations), encoding="utf-8")
(run_dir / "COMPLETE").write_text(created.isoformat(), encoding="utf-8")
print("Thesis archive:", run_dir.resolve())
print("Use only archives with a COMPLETE marker; source confirmation:", dedalus_source_confirmed)


### SGS flux diagnostics

Run after second-order discovery and the thesis archive cell. Uses Pi_tau = -S_ij tau_ij, S_ij = (d_i ubar_j + d_j ubar_i)/2, and Pi_phi = -phi_i d_i bbar. The notebook scalar b is buoyancy; interpreting it as T uses the simulation normalization. Full stress contractions include both off-diagonal entries and the trace contribution. Positive Pi denotes forward transfer and negative Pi denotes backscatter.

Candidate 0 is the first-order baseline. All trace/STF candidate pairs within each baseline mode are evaluated explicitly. Errors, correlation, signed mean transfers, negative fractions, and sign agreement use the same interior points for measured and modeled fields. These are diagnostics on discovery data, not independent validation. Spatial finite differences can amplify noise in second-order terms; eight boundary points are discarded. No claim of derivative convergence is made. Run on PACE; the cells stream frames and retain summaries plus a downsampled middle-frame flux snapshot. Save this notebook before archiving.


In [ ]:
# Flux diagnostics: run after second-order discovery and the thesis archive cell.
from copy import copy
from itertools import product
from SPIDER.commons.process_library_terms import IntegrationDomain
from SPIDER.commons.z3base import LiteralIndex, highest_index

# b is the simulated buoyancy/temperature-like scalar. No dimensional T conversion.
# All frames, same interior crop, equal spatial weights (uniform analysis grid).
flux_margin = 8  # discard high-order finite-difference boundary stencils
flux_snapshot_stride = 4
flux_sign_tolerance = 0.0  # absolute Pi units; increase to ignore near-zero signs

def flux_project(value, rank):
    if rank == 0:
        return value[..., None, None] * np.eye(2) / 2
    if rank == 2:
        symmetric = (value + value.swapaxes(-1, -2)) / 2
        return symmetric - np.trace(symmetric, axis1=-2, axis2=-1)[..., None, None] * np.eye(2) / 2
    return value

def flux_term_frame(dataset, term, it):
    """Use SPIDER's literal-index evaluator, summing every dummy index."""
    shape = tuple(int(n) for n in dataset.world_size[:2])
    domain = IntegrationDomain([0, 0, it], [shape[0]-1, shape[1]-1, it])
    result = np.zeros(shape + (2,) * term.rank)
    count = highest_index(term.all_indices()) + 1
    if any(prime.derivative.torder for prime in term.primes):
        raise ValueError("Flux evaluator requires spatial-only closure terms")
    for assignment in product(range(2), repeat=count):
        literal = term.map_all_indices(lambda idx: LiteralIndex(assignment[idx.value])) if count else term
        result[(slice(None), slice(None)) + assignment[:term.rank]] += dataset.eval_term(literal, domain)[..., 0]
    return result

def flux_statistics(reference, predicted):
    a, b = reference.ravel(), predicted.ravel()
    if not np.isfinite(a).all() or not np.isfinite(b).all():
        raise ValueError("Nonfinite flux; diagnose fields/derivatives before reporting")
    active = np.abs(a) > flux_sign_tolerance
    return np.array([a.size, a.sum(), b.sum(), a@a, b@b, a@b,
                     (a-b)@(a-b), np.sum(a < -flux_sign_tolerance),
                     np.sum(b < -flux_sign_tolerance), active.sum(),
                     np.sum(active & (np.sign(a) == np.sign(b))),
                     np.maximum(a, 0).sum(), np.minimum(a, 0).sum(),
                     np.maximum(b, 0).sum(), np.minimum(b, 0).sum()], dtype=float)

def flux_summary(s):
    n, sa, sb, aa, bb, ab, ee, na, nb, active, agree, pa, ma, pb, mb = s
    va, vb = max(0., aa-sa*sa/n), max(0., bb-sb*sb/n)
    return dict(relative_l2_error=float(np.sqrt(ee/aa)) if aa > 0 else None,
                rmse=float(np.sqrt(ee/n)), mean_reference=float(sa/n),
                mean_model=float(sb/n), mean_bias=float((sb-sa)/n),
                correlation=float(np.clip((ab-sa*sb/n)/np.sqrt(va*vb), -1, 1)) if va*vb > 0 else None,
                negative_fraction_reference=float(na/n), negative_fraction_model=float(nb/n),
                sign_agreement=float(agree/active) if active else None,
                positive_transfer_reference=float(pa/n), negative_transfer_reference=float(ma/n),
                positive_transfer_model=float(pb/n), negative_transfer_model=float(mb/n))

# Copy datasets shallowly so pointwise evaluation cannot fill discovery caches.
flux_datasets = {}
for target, source in second_order_datasets.items():
    evaluator = copy(source)
    evaluator.cache_primes = False
    flux_datasets[target] = evaluator

fx = np.asarray(x)[pad_x:len(x)-pad_x]
fy = np.asarray(y)[pad_y:len(y)-pad_y]
if min(len(fx), len(fy)) <= 2*flux_margin or flux_margin < 8:
    raise ValueError("Need an interior beyond the fourth-derivative, accuracy-6 stencils")
for coordinates, spacing in ((fx, dx), (fy, dy)):
    if not np.allclose(np.diff(coordinates), spacing):
        raise ValueError("SPIDER pointwise derivatives require the uniform analysis grid")
if resolved_fit['u'].shape[:3] != tau_fit.shape[:3] or phi_fit.shape[:3] != tau_fit.shape[:3]:
    raise ValueError("Unaligned closure fields")
if tau_fit.shape[:3] != (len(fx), len(fy), len(t)):
    raise ValueError("Coordinates do not match fitted fields")
cut = (slice(flux_margin, -flux_margin),) * 2

# Explicitly enumerate all trace/STF pairs; no silent choice of a thesis model.
flux_specs = []
for mode in sorted({key[2] for key in second_order_results}):
    groups = {}
    for key, entry in second_order_results.items():
        if key[2] != mode:
            continue
        rank = entry['problem'].terms[0].rank
        groups[key[0], rank] = [(key, index) for index in range(len(entry['fits']))]
    for pair in product(groups['tau', 0], groups['tau', 2]):
        flux_specs.append(('tau', list(pair)))
    for candidate in groups['phi', 1]:
        flux_specs.append(('phi', [candidate]))

flux_rows = []
flux_series = {str(i): [] for i in range(len(flux_specs))}
flux_sums = {str(i): np.zeros(15) for i in range(len(flux_specs))}
flux_snapshots = {}
flux_field_errors = {str(i): np.zeros(2) for i in range(len(flux_specs))}
for it in range(len(t)):
    velocity = np.asarray(resolved_fit['u'][:, :, it], dtype=float)
    scalar = np.asarray(resolved_fit['b'][:, :, it], dtype=float)
    # D[..., i, j] = d_i u_j; both off-diagonal contributions are included.
    D = np.stack(np.gradient(velocity, fx, fy, axis=(0, 1), edge_order=2), axis=-2)
    strain = ((D + D.swapaxes(-1, -2))/2)[cut]
    grad_b = np.stack(np.gradient(scalar, fx, fy, axis=(0, 1), edge_order=2), axis=-1)[cut]
    measured = {'tau': tau_fit[:, :, it][cut], 'phi': phi_fit[:, :, it][cut]}
    references = {'tau': -np.einsum('...ij,...ij->...', strain, measured['tau']),
                  'phi': -np.einsum('...i,...i->...', grad_b, measured['phi'])}
    # Cache only this frame's active terms and projected candidates.
    terms, candidates = {}, {}
    for key, entry in second_order_results.items():
        target = key[0]
        problem = entry['problem']
        rank = problem.terms[0].rank
        base = flux_datasets[target].data_dict[problem.gradient_name][:, :, it]
        if rank == 0:
            base = np.trace(base, axis1=-2, axis2=-1)
        for index, fit in enumerate(entry['fits']):
            value = problem.coefficient * np.asarray(base, dtype=float)
            for term, coefficient in zip(problem.terms[1:], fit['coefficients']):
                if coefficient == 0:
                    continue
                cache_key = (target, term)
                if cache_key not in terms:
                    terms[cache_key] = flux_term_frame(flux_datasets[target], term, it)
                value = value + coefficient * terms[cache_key]
            candidates[key, index] = flux_project(value[cut], rank)
    for index, (target, pieces) in enumerate(flux_specs):
        model = sum(candidates[piece] for piece in pieces)
        predicted = (-np.einsum('...ij,...ij->...', strain, model) if target == 'tau'
                     else -np.einsum('...i,...i->...', grad_b, model))
        label = str(index)
        flux_sums[label] += flux_statistics(references[target], predicted)
        flux_series[label].append([float(references[target].mean()), float(predicted.mean())])
        flux_field_errors[label] += [np.sum((model-measured[target])**2), np.sum(measured[target]**2)]
        if it == len(t)//2:
            flux_snapshots[label] = predicted[::flux_snapshot_stride, ::flux_snapshot_stride]
    if it == len(t)//2:
        for target, values in references.items():
            flux_snapshots['reference_'+target] = values[::flux_snapshot_stride, ::flux_snapshot_stride]
    print(f'Flux diagnostics frame {it+1}/{len(t)}', flush=True)

for index, (target, pieces) in enumerate(flux_specs):
    numerator, denominator = flux_field_errors[str(index)]
    flux_rows.append(dict(id=str(index), target=target,
        model_ids=[f'{key[0]}|{key[1]}|{key[2]}|candidate={candidate}' for key, candidate in pieces],
        closure_relative_l2_error=float(np.sqrt(numerator/denominator)) if denominator else None,
        **flux_summary(flux_sums[str(index)])))
for row in flux_rows:
    print(row)


In [ ]:
# Archive the diagnostics with the complete discovery provenance and candidate models.
# Run the thesis archive cell immediately before this section (save notebook first).
# Plotting is in the final section.

archived_models = {model['model_id']: model for model in bundle['models']}
for key, entry in second_order_results.items():
    for index, model in enumerate(summarize_corrections(entry['problem'], entry['fits'])):
        model_id = f'{key[0]}|{key[1]}|{key[2]}|candidate={index}'
        if model_id not in archived_models or any(archived_models[model_id][field] != model[field]
                for field in ('first_order_coefficient', 'correction_coefficients', 'complete_model')):
            raise ValueError('Discovery changed: rerun the thesis archive cell before flux archiving')

flux_created = datetime.now(timezone.utc)
flux_dir = archive_root / (flux_created.strftime('%Y%m%dT%H%M%S.%fZ') + '_flux_' + uuid4().hex[:8])
flux_dir.mkdir(parents=True, exist_ok=False)
flux_report = {
    'schema_version': 1, 'created_utc': flux_created.isoformat(),
    'validation_status': 'pointwise diagnostics on discovery data; not independent validation',
    'definitions': {'Pi_tau': '-sum_ij S_ij tau_ij', 'S_ij': '(d_i ubar_j + d_j ubar_i)/2',
                    'Pi_phi': '-sum_i phi_i d_i bbar'},
    'scalar_convention': 'b is the loaded buoyancy scalar; identified with T only in the simulation normalization',
    'sign_convention': 'positive transfer removes resolved energy/scalar variance; negative is backscatter',
    'settings': {'spatial_margin': flux_margin, 'snapshot_stride': flux_snapshot_stride,
                 'sign_tolerance': flux_sign_tolerance, 'time_indices': list(range(len(t))),
                 'averaging': 'equal uniform-grid spatial weights and equal snapshot weights',
                 'resolved_gradients': 'numpy.gradient, centered second order on interior',
                 'candidate_derivatives': 'SPIDER eval_term/FinDiff accuracy=6, spatial only',
                 'tau_assembly': 'all Cartesian pairs of trace and STF candidates within each baseline mode',
                 'first_order_models': 'candidate=0 has zero correction in each irrep',
                 'crop': 'resolved_fit / tau_fit / phi_fit, then spatial_margin on each side'},
    'discovery_archive': str(run_dir), 'discovery_bundle': deepcopy(bundle),
    'current_git': {'commit': git_metadata('rev-parse', 'HEAD'), 'status': git_metadata('status', '--porcelain')},
    'results': flux_rows,
    'executed_helpers': {},
}
for name in ('flux_project', 'flux_term_frame', 'flux_statistics', 'flux_summary'):
    try:
        flux_report['executed_helpers'][name] = inspect.getsource(globals()[name])
    except (OSError, TypeError):
        flux_report['executed_helpers'][name] = None
# Preserve the parent archive's source snapshots so this archive is self-contained.
for source_file in [*run_dir.glob('source_*'), *run_dir.glob('input_source_*'), run_dir / 'input_coordinates.npz']:
    (flux_dir / source_file.name).write_bytes(source_file.read_bytes())
notebook_path = Path('COnvection-FILTER-Copy2.ipynb')
current_notebook = json.loads(notebook_path.read_text(encoding='utf-8'))
(flux_dir / 'flux_notebook_source.json').write_text(json.dumps({
    'cells': [{'cell_type': c['cell_type'], 'source': c['source']} for c in current_notebook['cells']]
}, indent=2), encoding='utf-8')
np.savez_compressed(flux_dir / 'flux_arrays.npz', t=np.asarray(t),
    x=fx[flux_margin:-flux_margin], y=fy[flux_margin:-flux_margin],
    **{'mean_'+key: np.asarray(value) for key, value in flux_series.items()},
    **{'snapshot_'+key: value for key, value in flux_snapshots.items()})
(flux_dir / 'flux.json').write_text(json.dumps(flux_report, indent=2, allow_nan=False,
    default=archive_json_default), encoding='utf-8')
with (flux_dir / 'flux.csv').open('w', newline='', encoding='utf-8') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(flux_rows[0]))
    writer.writeheader()
    writer.writerows(flux_rows)

(flux_dir / 'COMPLETE').write_text(flux_created.isoformat(), encoding='utf-8')
print('Flux archive:', flux_dir.resolve())


### Comparison with the local SGS paper

Reference: Ugliotti et al., *Physics-informed data-driven inference of an interpretable equivariant LES model of two-dimensional incompressible fluid turbulence*, local draft `2D_Incompressible_SGS_Paper_2 (44).pdf`.

- Section 3.1, p. 6, equations (3.1) and (3.2): first fit the gradient-product stress, then fit the residual. The second correction is the contracted product of velocity Hessians.
- Equation (3.2), Appendix B equation (B6), and Appendix C equation (C4): the Gaussian NGM4 coefficients are `C1 = delta**2 / 12` and `C2 = delta**4 / 288`. The paper calls these order-2 and order-4 terms in filter width; our first/second discovery stages correspond to those two terms.
- In the output, choose `tau | Rank 0 | theoretical` or `tau | Symmetric trace-free rank 2 | theoretical`, then the row `terms = 1`. The line beginning `R[...]` is the correction alone; the next line beginning `tau_...` is the complete first-plus-second model. Each equation refers to the displayed trace/STF projection, not an independently fitted full tensor.
- For the posted run, C1 = 0.000198682149251 gives C2 = 1.97372982155e-8. The one-correction trace/STF fits give 2.55314539781e-8 / 2.39778230186e-8 (29.4% / 21.5% above theory). Their unscaled weak training errors fall from 19.0% / 21.8% to 6.1% / 7.8%. This supports the structure, not exact coefficient recovery.
- The analogous phi Hessian-product fit gives 2.62200397754e-8 (32.8% above theory), reducing error from 39.6% to 11.6%. This is an extension to buoyancy flux; the paper does not discover this Rayleigh-Benard scalar-flux closure.
- The paper varies filter scale and viscosity to establish coefficient scaling. A single run cannot establish universality. Its reference flows are doubly periodic, decaying 2D turbulence; our buoyancy-driven, vertically bounded convection differs.
- The paper's full **NGMR** model additionally introduces an unresolved stress field and its evolution equation: equations (3.4) and (3.7), pp. 7-8. The present residual fits do not constitute that full model.
- The paper's accuracy measure in (3.8)-(3.9) is a magnitude-aware correlation, not our unscaled weak relative error. Numerical error percentages are not directly comparable to its figures.

There is a candidate model for each sparsity, target, irrep, and baseline choice. The smallest training error does not automatically identify the model to use in the thesis.


### Running on PACE

Use the same Python 3.12+ SPIDER environment as the interactive notebook. `python run_filter_sweep.py --help` lists the scale/resource controls. Each process handles one filter scale with fresh arrays and a unique output directory. The Slurm array selects one width from `FILTER_SCALES`; failures retain their logs and intermediate reports and do not create the top-level COMPLETE marker.

The setup archives current source snapshots and inferred Dedalus configuration. Confirm the data's actual simulation provenance separately. Known-balance checks report errors with and without SGS divergence; candidate recovery uses measured tau/phi. Closure errors, first-order comparisons, second-order candidates, and optional flux diagnostics are saved separately with their equations/settings. Consult PACE_FILTER_SWEEP.md for details.


### Optional plots

All numerical work and archives are complete before these cells. The batch runner uses a headless backend and enables plots only with --plots.


In [ ]:
# All plots are optional and run after the numerical reports and archives.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for axis, field in zip(axes, ('u', 'b')):
    report = recovery_report['equations'][field]
    axis.semilogy([r['requested_terms'] for r in report['candidates']],
                  [r['target_relative_error'] for r in report['candidates']], 'o-', label='recovered')
    error = report['known']['target_relative_error']
    if error is not None:
        axis.axhline(error, color='black', label='known balance incl. SGS')
    axis.set(xlabel='requested nonzero terms (including target)', ylabel='relative weak error',
             title='Momentum' if field == 'u' else 'Heat')
    axis.legend()
fig.savefig(analysis_run_dir / 'recovery_errors.png', dpi=180)
plt.show()
plt.close(fig)

# Pointwise balance maps at one interior time, complementing weak-form checks.
it = len(t)//2
velocity, scalar, pressure = u_bar[:, :, it], b_filt[:, :, it], p_filt[:, :, it]
ux, uy = np.gradient(velocity, x, y, axis=(0, 1), edge_order=2)
bx, by = np.gradient(scalar, x, y, axis=(0, 1), edge_order=2)
px, py = np.gradient(pressure, x, y, axis=(0, 1), edge_order=2)
lap_u = np.gradient(ux, x, axis=0, edge_order=2)+np.gradient(uy, y, axis=1, edge_order=2)
lap_b = np.gradient(bx, x, axis=0, edge_order=2)+np.gradient(by, y, axis=1, edge_order=2)
div_tau = (np.gradient(tau[:, :, it, :, 0], x, axis=0, edge_order=2)
           + np.gradient(tau[:, :, it, :, 1], y, axis=1, edge_order=2))
div_phi = (np.gradient(phi[:, :, it, 0], x, axis=0, edge_order=2)
           + np.gradient(phi[:, :, it, 1], y, axis=1, edge_order=2))
ut = (u_bar[:, :, it+1]-u_bar[:, :, it-1])/(t[it+1]-t[it-1])
bt = (b_filt[:, :, it+1]-b_filt[:, :, it-1])/(t[it+1]-t[it-1])
momentum = (ut+velocity[..., 0, None]*ux+velocity[..., 1, None]*uy
            + np.stack([px, py], axis=-1)-nu*lap_u+div_tau)
momentum[..., 1] -= scalar
heat = bt+velocity[..., 0]*bx+velocity[..., 1]*by-kappa*lap_b+div_phi
fig, axes = plt.subplots(3, 1, figsize=(12, 8), constrained_layout=True)
for axis, values, title in zip(axes, (momentum[..., 0], momentum[..., 1], heat),
                               ('Horizontal momentum', 'Vertical momentum', 'Heat')):
    im = axis.imshow(values[2:-2, 2:-2].T, origin='lower', cmap='seismic',
        extent=[x[2], x[-3], y[2], y[-3]], aspect='equal')
    axis.set(title=f'{title} pointwise residual, t={t[it]:.6g}', xlabel='x', ylabel='y')
    fig.colorbar(im, ax=axis)
fig.savefig(analysis_run_dir / 'pointwise_balances.png', dpi=180)
plt.show()
plt.close(fig)

it = len(t)//2
fig, axes = plt.subplots(2, 1, figsize=(12, 6), constrained_layout=True)
for axis, values, title, xc, yc in (
    (axes[0], U[:, :, it], 'Raw horizontal velocity', x_raw, y_raw),
    (axes[1], u_bar[:, :, it, 0], 'Filtered horizontal velocity', x, y)):
    im = axis.imshow(values.T, origin='lower', extent=[xc[0], xc[-1], yc[0], yc[-1]], aspect='equal', cmap='RdBu_r')
    axis.set(title=title, xlabel='x', ylabel='y')
    fig.colorbar(im, ax=axis)
fig.savefig(analysis_run_dir / 'velocity.png', dpi=180)
plt.show()
plt.close(fig)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), constrained_layout=True)
for axis, u, v, xc, yc, title in (
    (axes[0], U[:, :, it], V[:, :, it], x_raw, y_raw, 'Raw continuity'),
    (axes[1], u_bar[:, :, it, 0], u_bar[:, :, it, 1], x, y, 'Filtered continuity')):
    divergence = np.gradient(u, xc, axis=0, edge_order=2)+np.gradient(v, yc, axis=1, edge_order=2)
    im = axis.imshow(divergence[1:-1, 1:-1].T, origin='lower', cmap='seismic',
        extent=[xc[1], xc[-2], yc[1], yc[-2]], aspect='equal')
    axis.set(title=title, xlabel='x', ylabel='y')
    fig.colorbar(im, ax=axis)
    print(title, 'interior RMS:', np.sqrt(np.mean(divergence[1:-1, 1:-1]**2)))
fig.savefig(analysis_run_dir / 'continuity.png', dpi=180)
plt.show()
plt.close(fig)


In [ ]:
if 'flux_rows' in globals():
    # Plot every candidate's spatial mean. IDs map directly to flux.csv and flux.json.
    fig, axes = plt.subplots(2, 1, figsize=(11, 8), constrained_layout=True)
    for axis, target in zip(axes, ('tau', 'phi')):
        rows = [row for row in flux_rows if row['target'] == target]
        axis.plot(t, np.asarray(flux_series[rows[0]['id']])[:, 0], color='black', lw=2, label='measured')
        for row in rows:
            axis.plot(t, np.asarray(flux_series[row['id']])[:, 1], alpha=.6, label=row['id'])
        axis.axhline(0, color='gray', lw=.5)
        axis.set(xlabel='simulation time', ylabel=f'mean Pi_{target}')
        axis.legend(ncol=6, fontsize=6)
    fig.savefig(flux_dir / 'mean_fluxes.png', dpi=180)
    plt.show()
    plt.close(fig)
